# Notebook 08 — GNNs for Molecular Biology

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 8 of 12  
**Time estimate:** 75 minutes

> A molecule is a graph: atoms are nodes, bonds are edges.
> Graph-level property prediction — is this molecule toxic? soluble? active? —
> is one of the most practically impactful applications of GNNs, and it
> directly extends the node-level GCN from NB07.

---
## Step 1 — Motivation

Traditional drug property prediction used fixed fingerprints (Morgan/ECFP)
as features for ML. GNNs learn the featurization from data — they learn which
substructures matter for the property of interest. ChemProp (Yang 2019) showed
that message-passing GNNs outperform ECFP+RF on 19 of 30 MoleculeNet benchmarks.
Understanding the architecture lets you apply it to any molecular prediction task.

---
## Step 2 — Intuition

**Molecular graph:**
- Nodes = atoms (features: element type, formal charge, degree, aromaticity,
  hybridization, ring membership, hydrogen count)
- Edges = bonds (features: bond type single/double/triple/aromatic, in ring,
  conjugated)
- No fixed position — the graph captures topology, not 3D geometry

**Graph-level prediction:**
1. Run $k$ GNN layers to get node embeddings $h_v^{(k)}$
2. **Graph pooling** (readout): aggregate all node embeddings into one vector
   - Mean pooling: $h_G = \frac{1}{|V|}\sum_v h_v^{(k)}$
   - Sum pooling: $h_G = \sum_v h_v^{(k)}$ (better for counting substructures)
   - Attention pooling: learned weights over nodes
3. FC layers on $h_G$ → prediction

**ECFP (Morgan fingerprints) — the traditional baseline:**
Extend circular neighborhoods of radius $r$ around each atom.
Hash the neighborhood identifiers into a bit vector.
ECFP4 (radius 2) captures 4-bond neighborhoods — very similar to a 2-layer GNN
with sum aggregation, but with a fixed (non-learned) hash function.

---
## Step 3 — Biological Background

**ChemProp (Yang 2019, JCIM):**
Directed message-passing: separate messages for each directed edge.
$m_{vw} = \sum_{u \in \mathcal{N}(v) \setminus w} h_{uv}$
Prevents information looping back from source to sender.
Achieves best-in-class on MoleculeNet.

**MoleculeNet benchmark (Wu 2018):**
Standard benchmark for molecular ML. Includes:
- BACE: beta-secretase inhibition (Alzheimer's drug target)
- ESOL: aqueous solubility
- HIV: HIV replication inhibition
- Tox21: 12-task toxicity panel

**SE(3)-equivariant GNNs:**
For 3D molecular structure (not just connectivity), GNNs must respect
rotational and translational symmetry. E(3)-equivariant networks:
SchNet, DimeNet, PaiNN, EGNN, NequIP.
Used in AlphaFold's structure module and protein design models.

**SMILES as string vs. molecular graph:**
SMILES encodes molecules as strings (e.g., `CC(=O)O` = acetic acid).
Older approaches used LSTM/transformers on SMILES strings (ChemBERTa).
GNNs on molecular graphs are generally better because they explicitly
represent connectivity without the artifacts of linear SMILES notation.

---
## Step 4 — Mathematical Explanation

**Message-passing GNN (general form):**
$$m_v^{(l+1)} = \text{AGGREGATE}\left(\{\text{MSG}(h_v^{(l)}, h_u^{(l)}, e_{uv}) : u \in \mathcal{N}(v)\}\right)$$
$$h_v^{(l+1)} = \text{UPDATE}\left(h_v^{(l)}, m_v^{(l+1)}\right)$$

**For molecular property prediction (sum aggregation, GIN-style):**
$$h_v^{(l+1)} = \text{MLP}\left((1 + \epsilon) h_v^{(l)} + \sum_{u \in \mathcal{N}(v)} h_u^{(l)}\right)$$

GIN (Graph Isomorphism Network, Xu 2019) is maximally expressive among the
message-passing GNN class — it can distinguish any two non-isomorphic graphs
that the Weisfeiler-Lehman graph isomorphism test can distinguish.

**Graph readout:**
$$h_G = \text{MLP}\left(\sum_{l=0}^{K} \text{POOL}(\{h_v^{(l)} : v \in V\})\right)$$

Concatenating representations from all layers (JUMPING KNOWLEDGE) before pooling
helps capture both local and global structure.

In [ ]:
# Step 6 — Python: GNN for molecular property prediction
# We simulate molecular graphs since RDKit is not always available.
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, mean_squared_error

torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Simulate molecular graphs ----
# Each molecule: a connected graph of 5-15 atoms
# Node features: [element_1hot(C,N,O,S,F), degree, aromatic]
# ELEMENTS: C=0,N=1,O=2,S=3,F=4  (5 elements)
ELEMENTS = ['C','N','O','S','F']
N_ELEM = 5
ATOM_FEAT_DIM = N_ELEM + 2  # element onehot + degree + aromatic

rng = np.random.default_rng(42)
N_MOLS = 800

def simulate_molecule(mol_id, rng):
    """Simulate a small molecule as adjacency + features."""
    n_atoms = rng.integers(6, 15)
    # Randomly generate a connected graph (path + extra edges)
    edges = []
    for i in range(1, n_atoms):
        j = rng.integers(0, i)
        edges.append((i, j)); edges.append((j, i))
    # Add a few extra bonds
    for _ in range(rng.integers(0, n_atoms//2)):
        u, v = rng.choice(n_atoms, 2, replace=False)
        if (u,v) not in edges:
            edges.append((u,v)); edges.append((v,u))
    
    # Node features
    # Bias composition: oxygen/nitrogen-rich molecules tend to be more soluble
    elem_probs = [0.6, 0.15, 0.15, 0.05, 0.05]
    elem_types = rng.choice(N_ELEM, n_atoms, p=elem_probs)
    degrees = np.bincount([u for u,v in edges], minlength=n_atoms)
    aromatic = (rng.random(n_atoms) < 0.3).astype(float)
    
    X_atom = np.zeros((n_atoms, ATOM_FEAT_DIM), dtype=np.float32)
    for i, et in enumerate(elem_types):
        X_atom[i, et] = 1.0
    X_atom[:, N_ELEM] = degrees / 6.0  # normalize degree
    X_atom[:, N_ELEM+1] = aromatic
    
    # Property: logP proxy — correlates with C count, negatively with O count
    n_carbon = (elem_types == 0).sum()
    n_oxygen = (elem_types == 2).sum()
    n_nitrogen = (elem_types == 1).sum()
    n_aromatic = aromatic.sum()
    logP = 0.5*n_carbon - 0.8*n_oxygen - 0.3*n_nitrogen + 0.2*n_aromatic + rng.normal(0, 0.5)
    
    return {'X': X_atom, 'edges': edges, 'logP': float(logP), 'n_atoms': n_atoms}

molecules = [simulate_molecule(i, rng) for i in range(N_MOLS)]
logP_vals = np.array([m['logP'] for m in molecules])
print(f'Generated {N_MOLS} molecules')
print(f'logP range: [{logP_vals.min():.1f}, {logP_vals.max():.1f}], mean={logP_vals.mean():.1f}')

# ---- GNN model (graph-level regression) ----
class MoleculeGNN(nn.Module):
    def __init__(self, atom_feat_dim, hidden_dim=64, n_layers=3):
        super().__init__()
        self.input_proj = nn.Linear(atom_feat_dim, hidden_dim)
        self.conv_layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)
        ])
        self.readout_fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.readout_fc2 = nn.Linear(hidden_dim, 1)
    
    def message_pass(self, H, edge_index):
        """Simple sum aggregation: each node collects from neighbors."""
        n = H.shape[0]
        H_agg = torch.zeros_like(H)
        for u, v in edge_index:
            H_agg[v] += H[u]
        return H_agg
    
    def forward(self, X, edge_index):
        H = F.relu(self.input_proj(X))
        for conv in self.conv_layers:
            H_neigh = self.message_pass(H, edge_index)
            H = F.relu(conv(H + H_neigh))  # residual + aggregation
        # Global mean pooling
        h_graph = H.mean(dim=0)
        h_graph = F.relu(self.readout_fc1(h_graph))
        return self.readout_fc2(h_graph).squeeze()

# ---- ECFP-like baseline: just use atom composition counts ----
def molecule_to_ecfp_approx(mol):
    """Approximate fingerprint: element counts + degree histogram."""
    elem_counts = np.zeros(N_ELEM)
    for feat in mol['X']:
        elem_counts += feat[:N_ELEM]
    n_atoms = mol['n_atoms']
    n_bonds = len(mol['edges']) // 2
    return np.concatenate([elem_counts/n_atoms, [n_atoms/15, n_bonds/20, mol['X'][:,N_ELEM+1].mean()]])

X_ecfp = np.array([molecule_to_ecfp_approx(m) for m in molecules], dtype=np.float32)

# ---- Training ----
train_mols = molecules[:600]; val_mols = molecules[600:]
train_logP = logP_vals[:600]; val_logP = logP_vals[600:]

model_gnn = MoleculeGNN(ATOM_FEAT_DIM, hidden_dim=64, n_layers=3).to(device)
optimizer_gnn = optim.Adam(model_gnn.parameters(), lr=1e-3)
criterion_reg = nn.MSELoss()

gnn_train_losses, gnn_val_mses = [], []
for epoch in range(80):
    model_gnn.train()
    rng.shuffle(train_mols)  # shuffle training order
    bl = []
    for mol in train_mols:
        optimizer_gnn.zero_grad()
        X_m = torch.tensor(mol['X']).to(device)
        pred = model_gnn(X_m, mol['edges'])
        target = torch.tensor(mol['logP'], dtype=torch.float32).to(device)
        loss = criterion_reg(pred, target)
        loss.backward(); optimizer_gnn.step()
        bl.append(loss.item())
    gnn_train_losses.append(np.mean(bl))
    
    model_gnn.eval()
    with torch.no_grad():
        val_preds = [model_gnn(torch.tensor(m['X']).to(device), m['edges']).item()
                       for m in val_mols]
    gnn_val_mses.append(mean_squared_error(val_logP, val_preds))
    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1}: train loss={gnn_train_losses[-1]:.4f}, val MSE={gnn_val_mses[-1]:.4f}')

# ECFP baseline
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_ecfp[:600], train_logP)
ecfp_mse = mean_squared_error(val_logP, ridge.predict(X_ecfp[600:]))
print(f'\nGNN val MSE: {min(gnn_val_mses):.4f}')
print(f'ECFP+Ridge val MSE: {ecfp_mse:.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Training curves
ax = axes[0]
ax.plot(gnn_train_losses, 'steelblue', lw=1.5, label='Train loss')
ax2 = ax.twinx()
ax2.plot(gnn_val_mses, 'tomato', lw=1.5, label='Val MSE')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train MSE', color='steelblue')
ax2.set_ylabel('Val MSE', color='tomato')
ax.set_title('A. GNN training (logP prediction)\n(molecular graph regression)')
lines = [plt.Line2D([0],[0],color='steelblue',lw=2,label='Train'),
           plt.Line2D([0],[0],color='tomato',lw=2,label='Val')]
ax.legend(handles=lines, fontsize=8)

# Panel B: GNN predicted vs. true logP
ax = axes[1]
model_gnn.eval()
with torch.no_grad():
    all_preds = [model_gnn(torch.tensor(m['X']).to(device), m['edges']).item() for m in val_mols]
ax.scatter(val_logP, all_preds, alpha=0.6, s=20, color='steelblue')
lim = [min(val_logP.min(), min(all_preds))-0.5, max(val_logP.max(), max(all_preds))+0.5]
ax.plot(lim, lim, 'k--', lw=0.8)
ax.set_xlabel('True logP'); ax.set_ylabel('Predicted logP')
gnn_mse_final = mean_squared_error(val_logP, all_preds)
ax.set_title(f'B. GNN predicted vs. true logP\nMSE={gnn_mse_final:.3f}')

# Panel C: GNN vs. ECFP baseline MSE
ax = axes[2]
ax.bar(['ECFP approx\n+ Ridge', 'GNN\n(message passing)'],
         [ecfp_mse, gnn_mse_final], color=['steelblue', 'tomato'],
         edgecolor='black', alpha=0.85, width=0.5)
ax.set_ylabel('Validation MSE (lower = better)')
ax.set_title('C. GNN vs. fingerprint baseline\n(GNN learns relevant substructures)')
for i, v in enumerate([ecfp_mse, gnn_mse_final]):
    ax.text(i, v+0.02, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Module 14 NB08: GNNs for Molecular Biology', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('molecular_gnns.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement mean pooling and sum pooling for graph readout. Show that sum pooling
   is more sensitive to molecular size (number of atoms) — which is better for logP?
2. Add bond type as an edge feature (single=1, double=2). How would you incorporate
   edge features into the message passing? Implement it.
3. If RDKit is available: convert a real SMILES string to a molecular graph using
   `Chem.MolFromSmiles`. Extract atom features and bond features. Train your GNN
   on real ESOL solubility data.
4. What is the relationship between a 2-layer GNN with sum aggregation and Morgan
   circular fingerprints (ECFP4)? Explain the key difference.

---
## Step 10 — Quiz

1. Write the general message-passing update equation. What are the MSG, AGGREGATE,
   and UPDATE functions in a GCN?
2. Why is sum pooling preferred over mean pooling for counting molecular substructures?
3. What is the Weisfeiler-Lehman graph isomorphism test and why does it bound GNN
   expressiveness?
4. Explain the difference between graph-level, node-level, and edge-level prediction
   tasks with one biological example of each.
5. What is SE(3)-equivariance and why is it important for 3D molecular GNNs?

---
## Step 12 — Reflection

> *[In one paragraph: explain how a GNN for molecular property prediction learns
> a representation analogous to extended connectivity fingerprints (ECFP), and
> what critical advantage the learned GNN representation has over fixed fingerprints.]*

---
*Next: `09_training_best_practices.ipynb`*